# cognitive_alpha_external_provider_smoke_v1

qlib-native smoke card for external command-provider readiness in the CogAlpha goal runner.

In [1]:

from __future__ import annotations

import copy
import json
import sys
import tempfile
from pathlib import Path

import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if ROOT.as_posix() not in sys.path:
    sys.path.insert(0, ROOT.as_posix())

from src.data import QlibProviderConfig, build_alpha158_legacy_dataset, init_qlib_provider
from src.workflow import (
    CognitiveAlphaEvolutionSpec,
    build_cognitive_alpha_codex_exec_candidate_provider,
    build_cognitive_alpha_command_llm_candidate_provider,
    codex_cli_readiness,
    run_cognitive_alpha_evolution,
)

CARD = ROOT / "research" / "cognitive_alpha_external_provider_smoke_v1"
EXPERIMENT = "cognitive_alpha_external_provider_smoke_v1"
PROVIDER_URI = ROOT / "data" / "qlib" / "soxl"
RECORDER_URI = CARD / "qlib_recorder"
MANIFEST = json.loads((PROVIDER_URI / "qlib_store_manifest.json").read_text(encoding="utf-8"))
AS_OF = MANIFEST["calendar_end"]
TRAIN = ("2019-01-02", "2024-12-31")
VALID = ("2025-01-02", "2025-12-31")
TEST = ("2026-01-02", AS_OF)
PROVIDER_COMMAND = (sys.executable, (ROOT / "src" / "workflow" / "cognitive_alpha_fixture_provider.py").as_posix())
SUMMARY_PROVIDER_COMMAND = ["python", "-m", "src.workflow.cognitive_alpha_fixture_provider"]
CODEX_PROVIDER_COMMAND = ["codex", "exec", "--output-schema", "cognitive_alpha_candidate_schema.json"]
CODEX_EXECUTION_MODE = "codex_exec_chatgpt_auth_no_openai_api_key"

CARD.mkdir(parents=True, exist_ok=True)
RECORDER_URI.mkdir(parents=True, exist_ok=True)
config = QlibProviderConfig(provider_uri=PROVIDER_URI, region=MANIFEST["region"], market=MANIFEST["market"], kernels=1)
provider_meta = init_qlib_provider(config)  # qlib.init wrapper; keep qlib as the data system of record.

fields = ["$open", "$high", "$low", "$close", "$volume", "$vix"]
qlib_features = D.features(["SOXL"], fields, start_time=TRAIN[0], end_time=AS_OF, freq="day")
frame = qlib_features.reset_index().sort_values(["datetime", "instrument"]).reset_index(drop=True)
frame = frame.rename(columns={field: field.removeprefix("$") for field in fields})
frame["datetime"] = pd.to_datetime(frame["datetime"]).dt.normalize()
close = pd.to_numeric(frame["close"], errors="coerce")
frame["label_return_5d"] = close.pct_change(5).shift(-5).ffill().fillna(0.0)

alpha_dataset = build_alpha158_legacy_dataset(
    config,
    instruments=["SOXL"],
    label="Ref($close, -5) / $close - 1",
    train=TRAIN,
    valid=VALID,
    test=TEST,
)
alpha_frames = [alpha_dataset.prepare(segment, col_set="feature") for segment in ["train", "valid", "test"]]
alpha_frame = pd.concat(alpha_frames).reset_index()
alpha_frame["datetime"] = pd.to_datetime(alpha_frame["datetime"]).dt.normalize()
preferred_alpha = ["MA5", "MA10", "MA20", "MA30", "MA60", "MAX20", "MAX30", "VIX", "SOXL_RET_5", "VOLUME_RATIO_5"]
alpha_cols = [column for column in preferred_alpha if column in alpha_frame.columns]
alpha_subset = alpha_frame[["datetime", "instrument", *alpha_cols]].rename(
    columns={column: f"alpha158_{column}" for column in alpha_cols}
)
alpha158_columns = [f"alpha158_{column}" for column in alpha_cols]
frame = frame.merge(alpha_subset, on=["datetime", "instrument"], how="left")
mining_frame = frame.loc[frame["datetime"] <= pd.Timestamp(VALID[1])].copy()

codex_runtime = codex_cli_readiness(timeout_seconds=30.0)
codex_provider_builder = build_cognitive_alpha_codex_exec_candidate_provider

candidate_provider = build_cognitive_alpha_command_llm_candidate_provider(
    command=PROVIDER_COMMAND,
    provider_id="fixture_external_command_provider",
    model="fixture-command",
    timeout_seconds=30,
    openai_api_key_required=False,
    codex_direct_compatible=True,
    codex_execution_mode="fixture_schema_smoke_for_codex_exec_provider",
)

R.set_uri(RECORDER_URI.as_posix())
with tempfile.TemporaryDirectory(prefix="cogalpha_external_provider_") as tmp_dir:
    spec = CognitiveAlphaEvolutionSpec(
        run_id="cognitive_alpha_evolution",
        profile="soxl_smoke",
        runtime_dir=Path(tmp_dir),
        generations=2,
        parent_pool_size=3,
        children_per_generation=4,
        initial_pool_size=4,
        qualified_percentile=0.50,
        elite_percentile=0.75,
        max_abs_correlation=0.98,
        min_observations=180,
        max_missing_ratio=0.55,
        min_distinct_values=5,
        fallback_top_n=3,
    )
    with R.start(experiment_name=EXPERIMENT):
        recorder = R.get_recorder()
        R.log_params(
            provider_uri=provider_meta["provider_uri"],
            market=MANIFEST["market"],
            region=MANIFEST["region"],
            instrument="SOXL",
            train_start=TRAIN[0],
            train_end=TRAIN[1],
            valid_start=VALID[0],
            valid_end=VALID[1],
            test_start=TEST[0],
            test_end=TEST[1],
            profile=spec.profile,
            generations=spec.generations,
            parent_pool_size=spec.parent_pool_size,
            children_per_generation=spec.children_per_generation,
            initial_pool_size=spec.initial_pool_size,
            alpha158_feature_count=alpha_dataset.qlib_feature_set["feature_count"],
            alpha158_smoke_columns=len(alpha158_columns),
            candidate_provider=candidate_provider.provider_id,
            provider_command=" ".join(SUMMARY_PROVIDER_COMMAND),
            fixture_provider="true",
            codex_auth_mode=codex_runtime.get("auth_mode", "unknown"),
            codex_openai_api_key_required="false",
        )
        evolution = run_cognitive_alpha_evolution(
            mining_frame,
            spec=spec,
            label_col="label_return_5d",
            alpha158_columns=alpha158_columns,
            candidate_provider=candidate_provider,
        )
        runtime_run_dir = evolution.run_dir
        for checkpoint_json in [runtime_run_dir / "state.json", runtime_run_dir / "summary.json"]:
            payload = json.loads(checkpoint_json.read_text(encoding="utf-8"))
            payload["run_dir"] = "recorder:artifacts/cognitive_alpha_evolution"
            if "spec" in payload:
                payload["spec"]["runtime_dir"] = "recorder:artifacts"
            checkpoint_json.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
        for artifact in sorted(runtime_run_dir.rglob("*")):
            if artifact.is_file():
                rel_parent = artifact.relative_to(runtime_run_dir).parent.as_posix()
                artifact_path = "cognitive_alpha_evolution" if rel_parent == "." else f"cognitive_alpha_evolution/{rel_parent}"
                recorder.log_artifact(artifact.as_posix(), artifact_path=artifact_path)
        R.log_metrics(
            selected_count=float(evolution.summary["selected_count"]),
            repair_attempt_count=float(evolution.summary["repair_attempt_count"]),
            candidate_metric_rows=float(evolution.summary["candidate_metric_rows"]),
        )
        recorder_id = recorder.id
        recorder_local_dir = Path(recorder.get_local_dir())

for transient in ["code_cached.txt", "code_diff.txt", "code_status.txt"]:
    transient_path = recorder_local_dir / "artifacts" / transient
    if transient_path.exists():
        transient_path.unlink()

evolution_summary = copy.deepcopy(evolution.summary)
evolution_summary["run_dir"] = "recorder:artifacts/cognitive_alpha_evolution"
evolution_summary["spec"]["runtime_dir"] = "recorder:artifacts"
evolution_summary["candidate_provider"]["codex_exec_contract_compatible"] = True

sidecars = {
    "evolution_summary": "evolution_summary.json",
    "selected_features": "selected_features.csv",
    "failure_modes": "failure_modes.md",
}
(CARD / sidecars["evolution_summary"]).write_text(json.dumps(evolution_summary, indent=2, sort_keys=True), encoding="utf-8")
pd.DataFrame({"selected_feature": evolution.selected_feature_names}).to_csv(CARD / sidecars["selected_features"], index=False)
(CARD / sidecars["failure_modes"]).write_text(
    "# Failure Modes\n\n"
    "- 这是 Codex provider readiness smoke；Codex CLI 使用 ChatGPT 登录，不要求外部 API key。\n"
    "- 本卡用 fixture provider 快速证明 Codex exec schema、quality checker、repair loop 和 recorder wiring，未消耗完整 Codex 生成预算。\n"
    "- fixture provider 故意生成一个 future shift 候选，只用于证明 quality checker 和 repair loop 会进入 recorder。\n"
    "- SOXL 是单标的时间序列，弱于 CogAlpha 论文股票池横截面设定。\n"
    "- 本卡不生成生产买卖点，不构成买卖建议。\n",
    encoding="utf-8",
)
workflow_yaml = "\n".join(
    [
        f"experiment: {EXPERIMENT}",
        "card_type: diagnostic_cognitive_alpha_codex_provider_smoke",
        "paper_reference:",
        "  arxiv_id: '2511.18850'",
        "  method: codex_exec_provider_readiness_smoke",
        "qlib:",
        f"  provider_uri: {PROVIDER_URI.resolve().as_posix()}",
        f"  recorder_uri: {RECORDER_URI.resolve().as_posix()}",
        f"  market: {MANIFEST['market']}",
        f"  region: {MANIFEST['region']}",
        "segments:",
        f"  train: [{TRAIN[0]}, {TRAIN[1]}]",
        f"  validation: [{VALID[0]}, {VALID[1]}]",
        f"  test: [{TEST[0]}, {TEST[1]}]",
        "codex_provider:",
        "  provider_builder: build_cognitive_alpha_codex_exec_candidate_provider",
        f"  execution_mode: {CODEX_EXECUTION_MODE}",
        "  openai_api_key_required: false",
        "fixture_provider:",
        "  provider_id: fixture_external_command_provider",
        "  provider_kind: external_llm_code",
        "  model: fixture-command",
        "  command: python -m src.workflow.cognitive_alpha_fixture_provider",
        "evolution:",
        "  profile: soxl_smoke",
        "  generations: 2",
        "  initial_pool_size: 4",
        "  parent_pool_size: 3",
        "  children_per_generation: 4",
        "artifact_policy:",
        "  policy: slim_card_recorder_first",
        "  workflow_config: workflow.yaml",
        "  machine_entrypoint: summary.json",
        "",
    ]
)
(CARD / "workflow.yaml").write_text(workflow_yaml, encoding="utf-8")
recorder_artifacts = sorted(path.relative_to(recorder_local_dir).as_posix() for path in recorder_local_dir.rglob("*") if path.is_file())
summary = {
    "experiment": EXPERIMENT,
    "card_type": "diagnostic_cognitive_alpha_codex_provider_smoke",
    "workflow_config": "workflow.yaml",
    "promotion_decision": "not_promoted_codex_provider_readiness_only",
    "paper_reference": {
        "arxiv_id": "2511.18850",
        "version_checked": "v3",
        "method": "codex_exec_provider_readiness_smoke",
    },
    "provider_uri": PROVIDER_URI.resolve().as_posix(),
    "recorder_uri": RECORDER_URI.resolve().as_posix(),
    "recorder_id": recorder_id,
    "recorder_local_dir": recorder_local_dir.as_posix(),
    "recorder_artifacts": recorder_artifacts,
    "market": MANIFEST["market"],
    "region": MANIFEST["region"],
    "instrument": "SOXL",
    "as_of": AS_OF,
    "data_range": [TRAIN[0], AS_OF],
    "segments": {"train": list(TRAIN), "validation": list(VALID), "test": list(TEST)},
    "codex_provider_readiness": {
        **codex_runtime,
        "provider_builder": codex_provider_builder.__name__,
        "execution_mode": CODEX_EXECUTION_MODE,
        "provider_command": CODEX_PROVIDER_COMMAND,
        "fixture_provider_command": SUMMARY_PROVIDER_COMMAND,
        "real_codex_generation_run": False,
        "purpose": "Codex CLI readiness plus fixture schema smoke for prompt/response/repair/evolution/recorder wiring",
    },
    "qlib_native_api": [
        "qlib.init",
        "qlib.data.D.features",
        "DatasetH",
        "DataHandlerLP",
        "Alpha158.get_feature_config",
        "qlib.workflow.R.start",
    ],
    "qlib_docs": {
        "data": "https://qlib.readthedocs.io/en/latest/component/data.html",
        "workflow": "https://qlib.readthedocs.io/en/latest/component/workflow.html",
    },
    "alpha158_feature_set": alpha_dataset.qlib_feature_set,
    "alpha158_smoke_columns": alpha158_columns,
    "evolution": evolution_summary,
    "sidecars": sidecars,
    "artifact_policy": {
        "policy": "slim_card_recorder_first",
        "primary_artifact": f"{EXPERIMENT}.ipynb",
        "machine_entrypoint": "summary.json",
        "workflow_config": "workflow.yaml",
        "kept_sidecars": sorted(sidecars.values()),
        "recorder_only_artifacts": {
            "candidate_pool": "artifacts/cognitive_alpha_evolution/candidate_pool/generation_*.jsonl",
            "quality": "artifacts/cognitive_alpha_evolution/quality/generation_*.csv",
            "fitness": "artifacts/cognitive_alpha_evolution/fitness/generation_*.csv",
            "rejected": "artifacts/cognitive_alpha_evolution/rejected/generation_*.jsonl",
            "lineage": "artifacts/cognitive_alpha_evolution/lineage/generation_*.jsonl",
            "state": "artifacts/cognitive_alpha_evolution/state.json",
        },
    },
}
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
(CARD / "conclusion.md").write_text(
    "# 结论\n\n"
    "- what_ran: 执行 cognitive_alpha_external_provider_smoke_v1，用 qlib.init、D.features、DatasetH/DataHandlerLP、Alpha158.get_feature_config、R.start 和 Codex CLI readiness 跑通 Codex provider 合同 smoke。\n"
    f"- data: provider={PROVIDER_URI.resolve().as_posix()}，as_of={AS_OF}，train={TRAIN[0]} 到 {TRAIN[1]}，validation={VALID[0]} 到 {VALID[1]}，test={TEST[0]} 到 {TEST[1]}。\n"
    f"- key_metrics: codex_auth={codex_runtime.get('auth_mode', 'unknown')}，provider=fixture_external_command_provider，prompt_count={evolution_summary['candidate_provider']['prompt_count']}，response_count={evolution_summary['candidate_provider']['response_count']}，generations={evolution_summary['completed_generations']}，selected_count={evolution_summary['selected_count']}，repair_attempt_count={evolution_summary['repair_attempt_count']}。\n"
    "- regime/window: 本卡只验证 Codex exec provider schema、prompt/response provenance、质量检查、repair、五指标筛选、mutation/crossover、多代 checkpoint 和 recorder-first 证据。\n"
    "- promotion_decision: not_promoted_codex_provider_readiness_only。它不是生产模型，不生成最新买卖点。\n"
    "- risks: 本次为 fixture schema smoke，不是完整 Codex 候选池生成；SOXL 单标的 IC 近似弱于论文横截面股票池；本研究面向交易复盘和风险控制，不构成买卖建议。\n"
    f"- next_action: 后续直接用 Codex provider 扩展候选池，不再把外部 API key 作为运行前置条件；若要推广，仍需先超过 constrained_weight_reference、soxl_backtest。qlib recorder id={recorder_id}。\n",
    encoding="utf-8",
)

print(json.dumps({"recorder_id": recorder_id, "selected_count": evolution.summary["selected_count"], "as_of": AS_OF}, indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[22327:MainThread](2026-06-25 20:36:36,572) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[22327:MainThread](2026-06-25 20:36:36,575) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[22327:MainThread](2026-06-25 20:36:36,575) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[22327:MainThread](2026-06-25 20:36:36,591) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


...

[22327:MainThread](2026-06-25 20:36:36,656) INFO - qlib.timer - [log.py:127] - Time cost: 0.059s | Init data Done


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[22327:MainThread](2026-06-25 20:36:46,657) INFO - qlib.workflow - [exp.py:258] - Experiment 164629801568356971 starts running ...


[22327:MainThread](2026-06-25 20:36:46,725) INFO - qlib.workflow - [recorder.py:345] - Recorder 2791630ea7474fe99882dfb8fcbab947 starts running under Experiment 164629801568356971 ...


[22327:MainThread](2026-06-25 20:36:47,000) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{
  "recorder_id": "2791630ea7474fe99882dfb8fcbab947",
  "selected_count": 2,
  "as_of": "2026-06-24"
}


## 中文结论

详见 `conclusion.md`。本研究面向交易复盘和风险控制，不构成买卖建议。